# Deep Learning Forum 07: Text Classification & Preprocessing

Studi kasus: text classification berbasis dataset `train_data.csv`.

**Tujuan utama:**
1. Eksplorasi dataset text (EDA sederhana, distribusi label, contoh data)
2. Preprocessing text: case folding, cleaning, tokenisasi, stopword removal, stemming/lemmatisasi
3. Representasi text: one-hot, TF-IDF, dsb
4. Diskusi: mengapa preprocessing text itu kompleks?
5. Dokumentasi tantangan, solusi, dan rekomendasi

---

## 1. Setup Lingkungan & Import Library


In [58]:
# Import library utama
import pandas as pd
import numpy as np
import nltk
import re
import string
from nltk.tokenize import word_tokenize
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
print('Library utama berhasil di-import.')

Library utama berhasil di-import.


In [59]:
# Fungsi scraping Playstore (opsional)
from google_play_scraper import reviews, Sort, app
import time
import os
def fetch_reviews(package_name, lang='en', country='us', sort=Sort.MOST_RELEVANT, total=500, sleep=0.2):
    all_reviews = []
    count = 10
    next_token = None
    while len(all_reviews) < total:
        n = min(count, total - len(all_reviews))
        result, next_token = reviews(package_name, lang=lang, country=country, sort=sort, count=n, continuation_token=next_token)
        if not result:
            break
        all_reviews.extend(result)
        if not next_token:
            break
        time.sleep(sleep)
    return all_reviews
print('Fungsi scraping Playstore siap digunakan.')

Fungsi scraping Playstore siap digunakan.


In [60]:
# Contoh eksekusi scraping Playstore
pkg = 'com.king.candycrushsaga'
total_to_get = 100
meta = app(pkg, lang='id', country='id')
print(f"App: {meta['title']} - {meta['installs']}")
reviews_list = fetch_reviews(pkg, lang='en', country='us', sort=Sort.NEWEST, total=total_to_get)
print(f'Fetched {len(reviews_list)} reviews')
df_scraping = pd.DataFrame(reviews_list)
os.makedirs('output_data', exist_ok=True)
df_scraping.to_csv(f'output_data/{pkg}_reviews_games.csv', index=False, encoding='utf-8-sig')
print('Hasil scraping Playstore disimpan ke output_data/.')

App: Candy Crush Saga - 1.000.000.000+
Fetched 100 reviews
Hasil scraping Playstore disimpan ke output_data/.


In [61]:
print('Notebook siap digunakan untuk pipeline text classification, EDA, preprocessing, dan representasi.')

Notebook siap digunakan untuk pipeline text classification, EDA, preprocessing, dan representasi.


---

## 2. Exploratory Data Analysis (EDA)

Pada bagian ini dilakukan:
- Load dataset
- Lihat contoh data
- Distribusi label


In [62]:
# Load dataset utama dan hasil scraping jika ada
main_df = pd.read_csv('dataset/train_data.csv')
scraping_path = 'output_data/com.king.candycrushsaga_reviews_games.csv'
if os.path.exists(scraping_path):
    scraping_df = pd.read_csv(scraping_path)
    scraping_df = scraping_df.rename(columns={'content': 'text'})
    scraping_df['label'] = 0
    scraping_df = scraping_df[['text', 'label']]
    df = pd.concat([main_df, scraping_df], ignore_index=True)
    print(f'Dataset utama dan hasil scraping digabung. Jumlah data: {df.shape[0]}')
else:
    df = main_df
    print(f'Dataset utama digunakan. Jumlah data: {df.shape[0]}')
display(df.head())

Dataset utama dan hasil scraping digabung. Jumlah data: 17090


,text,label
0,Here are Thursday's biggest analyst calls: App...,0
1,Buy Las Vegas Sands as travel to Singapore bui...,0
2,"Piper Sandler downgrades DocuSign to sell, cit...",0
3,"Analysts react to Tesla's latest earnings, bre...",0
4,Netflix and its peers are set for a ‘return to...,0


In [63]:
# Distribusi label dan info data
print(f'Shape data: {df.shape}')
print('Distribusi label:')
print(df['label'].value_counts())

Shape data: (17090, 2)
Distribusi label:
label
2     3545
18    2118
14    1822
9     1557
5      987
16     985
1      837
19     823
7      624
6      524
15     501
17     495
12     487
13     471
4      359
0      355
3      321
8      166
10      69
11      44
Name: count, dtype: int64


---

## 3. Text Preprocessing

Langkah-langkah preprocessing yang dilakukan:
- Case folding
- Remove number
- Remove punctuation
- Remove multiple whitespace
- Tokenization
- Remove stopword
- Stemming/Lemmatization


![image.png](attachment:c57c84a4-b096-4fb2-b524-a7e375b3fdd0.png)

In [64]:
# --- Text Preprocessing Lengkap ---
import re, string, nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
import pandas as pd
import matplotlib.pyplot as plt
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('universal_tagset')

# Visualisasi fungsi pembanding sebelum-sesudah
def show_compare(before, after, title_before, title_after):
    df_vis = pd.DataFrame({title_before: before, title_after: after})
    display(df_vis.head())

# Case folding
df_lowercase = df['text'].str.lower()
print('Case folding selesai.')
show_compare(df['text'], df_lowercase, 'Original', 'Lowercase')
# Remove number
clean_text = [re.sub(r'\d+', '', i) for i in df_lowercase]
print('Remove number selesai.')
show_compare(df_lowercase, clean_text, 'Lowercase', 'No Number')
# Remove punctuation
clean_text_nopunct = [t.translate(str.maketrans('', '', string.punctuation)) for t in clean_text]
print('Remove punctuation selesai.')
show_compare(clean_text, clean_text_nopunct, 'No Number', 'No Punctuation')
# Remove multiple whitespace
clean_text_nowhitespace = [re.sub(r'\s+', ' ', t).strip() for t in clean_text_nopunct]
print('Remove multiple whitespace selesai.')
show_compare(clean_text_nopunct, clean_text_nowhitespace, 'No Punctuation', 'No Whitespace')
# Tokenization
tokens = [word_tokenize(t) for t in clean_text_nowhitespace]
print('Tokenisasi selesai.')
show_compare(clean_text_nowhitespace, tokens, 'No Whitespace', 'Tokens')
# Remove stopwords
stop_words = set(stopwords.words('english'))
tokens_nostop = [[w for w in tok if w not in stop_words] for tok in tokens]
print('Remove stopwords selesai.')
show_compare(tokens, tokens_nostop, 'Tokens', 'No Stopwords')
# Stemming
stemmer = PorterStemmer()
tokens_stem = [[stemmer.stem(w) for w in tok] for tok in tokens_nostop]
print('Stemming selesai.')
show_compare(tokens_nostop, tokens_stem, 'No Stopwords', 'Stemmed')
# Lemmatization
lemmatizer = WordNetLemmatizer()
tokens_lemma = [[lemmatizer.lemmatize(w) for w in tok] for tok in tokens_nostop]
print('Lemmatization selesai. Contoh:', tokens_lemma[:2])
# POS Tagging
def preprocess(sent):
    sent = nltk.word_tokenize(sent)
    sent = nltk.pos_tag(sent, lang='eng')
    return sent
sample_text = clean_text_nowhitespace[0] if len(clean_text_nowhitespace) > 0 else ''
print('POS Tagging contoh:', preprocess(sample_text))
# Named Entity Recognition (spaCy)
try:
    import spacy
    nlp = spacy.load('en_core_web_sm')
except OSError:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm'])
    import spacy
    nlp = spacy.load('en_core_web_sm')
doc = nlp('European authorities fined Google a record $5.1 billion on Wednesday for abusing its power in the mobile phone market and ordered the company to alter its practices')
print('Contoh NER:', [(X.text, X.label_) for X in doc.ents])

Case folding selesai.


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\yanli\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\yanli\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\yanli\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\yanli\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package universal_tagset to
[nltk_data]     C:\Users\yanli\AppData\Roaming\nltk_data...
[nltk_data]   Package universal_tagset is already up-to-date!


,Original,Lowercase
0,Here are Thursday's biggest analyst calls: App...,here are thursday's biggest analyst calls: app...
1,Buy Las Vegas Sands as travel to Singapore bui...,buy las vegas sands as travel to singapore bui...
2,"Piper Sandler downgrades DocuSign to sell, cit...","piper sandler downgrades docusign to sell, cit..."
3,"Analysts react to Tesla's latest earnings, bre...","analysts react to tesla's latest earnings, bre..."
4,Netflix and its peers are set for a ‘return to...,netflix and its peers are set for a ‘return to...


Remove number selesai.


,Lowercase,No Number
0,here are thursday's biggest analyst calls: app...,here are thursday's biggest analyst calls: app...
1,buy las vegas sands as travel to singapore bui...,buy las vegas sands as travel to singapore bui...
2,"piper sandler downgrades docusign to sell, cit...","piper sandler downgrades docusign to sell, cit..."
3,"analysts react to tesla's latest earnings, bre...","analysts react to tesla's latest earnings, bre..."
4,netflix and its peers are set for a ‘return to...,netflix and its peers are set for a ‘return to...


...

Remove stopwords selesai.


,Tokens,No Stopwords
0,"[here, are, thursdays, biggest, analyst, calls...","[thursdays, biggest, analyst, calls, apple, am..."
1,"[buy, las, vegas, sands, as, travel, to, singa...","[buy, las, vegas, sands, travel, singapore, bu..."
2,"[piper, sandler, downgrades, docusign, to, sel...","[piper, sandler, downgrades, docusign, sell, c..."
3,"[analysts, react, to, teslas, latest, earnings...","[analysts, react, teslas, latest, earnings, br..."
4,"[netflix, and, its, peers, are, set, for, a, ‘...","[netflix, peers, set, ‘, return, growth, ’, an..."


Stemming selesai.


,No Stopwords,Stemmed
0,"[thursdays, biggest, analyst, calls, apple, am...","[thursday, biggest, analyst, call, appl, amazo..."
1,"[buy, las, vegas, sands, travel, singapore, bu...","[buy, la, vega, sand, travel, singapor, build,..."
2,"[piper, sandler, downgrades, docusign, sell, c...","[piper, sandler, downgrad, docusign, sell, cit..."
3,"[analysts, react, teslas, latest, earnings, br...","[analyst, react, tesla, latest, earn, break, w..."
4,"[netflix, peers, set, ‘, return, growth, ’, an...","[netflix, peer, set, ‘, return, growth, ’, ana..."


Lemmatization selesai. Contoh: [['thursday', 'biggest', 'analyst', 'call', 'apple', 'amazon', 'tesla', 'palantir', 'docusign', 'exxon', 'amp', 'httpstcoqpngwluh'], ['buy', 'la', 'vega', 'sand', 'travel', 'singapore', 'build', 'well', 'fargo', 'say', 'httpstcoflswicz']]
POS Tagging contoh: [('here', 'RB'), ('are', 'VBP'), ('thursdays', 'JJ'), ('biggest', 'JJS'), ('analyst', 'NN'), ('calls', 'VBZ'), ('apple', 'NN'), ('amazon', 'NN'), ('tesla', 'NN'), ('palantir', 'NN'), ('docusign', 'NN'), ('exxon', 'VBZ'), ('amp', 'RB'), ('more', 'RBR'), ('httpstcoqpngwluh', 'JJ')]
Contoh NER: [('European', 'NORP'), ('Google', 'ORG'), ('$5.1 billion', 'MONEY'), ('Wednesday', 'DATE')]


In [65]:
import nltk
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\yanli\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

In [66]:
import nltk
nltk.download('wordnet')

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\yanli\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

### Contoh Kode Preprocessing Lengkap (dari text_preprocessing.ipynb)

In [67]:
# Case folding

df['text_lower'] = df['text'].str.lower()
df['text_lower'].head()

0    here are thursday's biggest analyst calls: app...
1    buy las vegas sands as travel to singapore bui...
2    piper sandler downgrades docusign to sell, cit...
3    analysts react to tesla's latest earnings, bre...
4    netflix and its peers are set for a ‘return to...
Name: text_lower, dtype: object

In [68]:
# Remove number

df['text_nonum'] = df['text_lower'].apply(lambda x: re.sub(r'\d+', '', x))
df['text_nonum'].head()

0    here are thursday's biggest analyst calls: app...
1    buy las vegas sands as travel to singapore bui...
2    piper sandler downgrades docusign to sell, cit...
3    analysts react to tesla's latest earnings, bre...
4    netflix and its peers are set for a ‘return to...
Name: text_nonum, dtype: object

In [69]:
# Remove punctuation

df['text_nopunct'] = df['text_nonum'].apply(lambda x: x.translate(str.maketrans('', '', string.punctuation)))
df['text_nopunct'].head()

0    here are thursdays biggest analyst calls apple...
1    buy las vegas sands as travel to singapore bui...
2    piper sandler downgrades docusign to sell citi...
3    analysts react to teslas latest earnings break...
4    netflix and its peers are set for a ‘return to...
Name: text_nopunct, dtype: object

In [70]:
# Remove multiple whitespace

df['text_nowhitespace'] = df['text_nopunct'].apply(lambda x: re.sub(r'\s+', ' ', x).strip())
df['text_nowhitespace'].head()

0    here are thursdays biggest analyst calls apple...
1    buy las vegas sands as travel to singapore bui...
2    piper sandler downgrades docusign to sell citi...
3    analysts react to teslas latest earnings break...
4    netflix and its peers are set for a ‘return to...
Name: text_nowhitespace, dtype: object

In [71]:
# Tokenization

df['tokens'] = df['text_nowhitespace'].apply(word_tokenize)
df['tokens'].head()

0    [here, are, thursdays, biggest, analyst, calls...
1    [buy, las, vegas, sands, as, travel, to, singa...
2    [piper, sandler, downgrades, docusign, to, sel...
3    [analysts, react, to, teslas, latest, earnings...
4    [netflix, and, its, peers, are, set, for, a, ‘...
Name: tokens, dtype: object

In [72]:
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\yanli\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [73]:
import nltk
nltk.download('punkt')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\yanli\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [74]:
# Remove stopwords (English)

nltk.download('stopwords')
from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))
df['tokens_nostop'] = df['tokens'].apply(lambda tokens: [w for w in tokens if w not in stop_words])
df['tokens_nostop'].head()

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\yanli\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


0    [thursdays, biggest, analyst, calls, apple, am...
1    [buy, las, vegas, sands, travel, singapore, bu...
2    [piper, sandler, downgrades, docusign, sell, c...
3    [analysts, react, teslas, latest, earnings, br...
4    [netflix, peers, set, ‘, return, growth, ’, an...
Name: tokens_nostop, dtype: object

In [75]:
# Stemming (PorterStemmer)

from nltk.stem import PorterStemmer
stemmer = PorterStemmer()
df['tokens_stem'] = df['tokens_nostop'].apply(lambda tokens: [stemmer.stem(w) for w in tokens])
df['tokens_stem'].head()

0    [thursday, biggest, analyst, call, appl, amazo...
1    [buy, la, vega, sand, travel, singapor, build,...
2    [piper, sandler, downgrad, docusign, sell, cit...
3    [analyst, react, tesla, latest, earn, break, w...
4    [netflix, peer, set, ‘, return, growth, ’, ana...
Name: tokens_stem, dtype: object

---
## 4. Representasi Teks Lanjutan: Word Embedding (GloVe & FastText)
Pada bagian ini, kita akan menggunakan pre-trained embedding populer: GloVe dan FastText, untuk merepresentasikan kata-kata dalam bentuk vektor numerik.

In [76]:
# --- Word2Vec: CBOW & Skipgram (Gensim) ---
import gensim
from gensim.models import Word2Vec
from gensim.models import FastText
import gensim.downloader as api
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
import numpy as np
import pandas as pd

# Tokenisasi ulang jika belum ada
if 'tokens' in df.columns:
    word_token = df['tokens'].tolist()
else:
    word_token = [word_tokenize(t) for t in df['text'].astype(str)]

# --- Word2Vec CBOW ---
model_cbow = Word2Vec(word_token, min_count=1, vector_size=100, window=5, sg=0)
vocabulary_cbow = model_cbow.wv.index_to_key
print('CBOW Vocabulary:', vocabulary_cbow[:10])
print('Similarity (amazon, apple):', model_cbow.wv.similarity('amazon','apple'))

# --- Word2Vec Skipgram ---
model_skipgram = Word2Vec(word_token, min_count=1, vector_size=100, window=5, sg=1)
vocabulary_skipgram = model_skipgram.wv.index_to_key
print('Skipgram Vocabulary:', vocabulary_skipgram[:10])
print('Similarity (amazon, apple):', model_skipgram.wv.similarity('amazon','apple'))

# --- GloVe (gensim downloader) ---
model_glove = api.load('glove-twitter-25')
print('GloVe most similar to amazon:', model_glove.most_similar(positive=['amazon'], topn=2))
vocabulary_glove = list(model_glove.key_to_index.keys())
vector_glove = model_glove.get_vector('amazon')
print('GloVe vector for amazon:', vector_glove[:5])

# --- FastText ---
meta_hyper = {
    'vector_size': 25, # size of embedding 
    'alpha': 0.025, # learning rate
    'window': 5,
    'min_freq': 1,
    'epochs': 3, # number of training epochs
    'CPU': 4
}
embed_model = FastText(vector_size=meta_hyper['vector_size'], window=meta_hyper['window'], min_count=meta_hyper['min_freq'], alpha=meta_hyper['alpha'], workers=meta_hyper['CPU'])
embed_model.build_vocab(word_token)
embed_model.train(word_token, total_examples=len(word_token), epochs=meta_hyper['epochs'])
vector_fasttext = embed_model.wv['amazon']
print('FastText vector for amazon:', vector_fasttext[:5])

CBOW Vocabulary: ['the', 'to', 'of', 'a', 'in', 'and', 'for', 'on', '’', 's']
Similarity (amazon, apple): 0.97814363
Skipgram Vocabulary: ['the', 'to', 'of', 'a', 'in', 'and', 'for', 'on', '’', 's']
Similarity (amazon, apple): 0.9189403
GloVe most similar to amazon: [('kindle', 0.8822218179702759), ('nexus', 0.8523538112640381)]
GloVe vector for amazon: [ 0.23029 -0.26417 -0.19669 -1.2001   0.84545]
FastText vector for amazon: [ 0.72218364 -0.7419887   0.4909945   0.1349816  -0.10994407]


---
## 5. Kritik, Saran, dan Kendala

**Kritik & Saran:**
- Visualisasi contoh sebelum dan sesudah pada setiap tahap preprocessing sangat membantu memahami efek transformasi data.
- Penambahan fungsi `show_compare` membuat proses validasi hasil preprocessing lebih transparan dan mudah dievaluasi.
- Notebook saya buat dengan terstruktur, namun dokumentasi singkat pada setiap blok kode tetap disarankan untuk memperjelas tujuan tiap langkah.

**Kendala yang Dihadapi:**
- Proses instalasi library eksternal (misal: `gensim`, `spacy`, model bahasa) tetap membutuhkan koneksi internet stabil dan waktu ekstra.
- Jika dataset sangat besar, visualisasi tabel bisa menjadi lambat, sehingga sebaiknya hanya menampilkan beberapa sampel saja.
- Beberapa data teks masih bisa mengandung karakter aneh atau format tidak terduga, sehingga perlu eksplorasi dan cleaning tambahan.

**Solusi:**
- Pastikan environment sudah lengkap dependensinya sebelum menjalankan pipeline.
- Untuk dataset besar, gunakan `df.sample(5)` pada visualisasi agar lebih efisien.

---
## 6. Jawaban Forum Diskusi (Update)

**Mengapa proses pre processing pada text memerlukan langkah-langkah yang cukup kompleks?**

Berdasarkan hasil notebook, setiap tahap preprocessing (case folding, remove number, remove punctuation, whitespace, tokenisasi, stopword removal, stemming) memberikan perubahan signifikan pada data. Visualisasi sebelum-sesudah membuktikan bahwa data teks sangat bervariasi dan penuh noise.

Kompleksitas preprocessing muncul karena:
- Data teks mentah mengandung banyak variasi, noise, dan ketidakteraturan.
- Setiap tahap (misal: stopword removal, stemming) mengurangi noise dan menyederhanakan data, sehingga model lebih mudah belajar.
- Tanpa preprocessing yang baik, model akan kesulitan mengenali pola karena data terlalu beragam.

Dengan visualisasi, kita bisa memastikan setiap langkah benar-benar membersihkan dan menstandarkan data, sehingga hasil analisis/model lebih akurat dan dapat diinterpretasikan.

In [79]:
# --- Generate Modern EDA HTML Otomatis (ydata-profiling) ---
from ydata_profiling import ProfileReport
import warnings
warnings.filterwarnings('ignore')
try:
    # Gunakan df hasil preprocessing/gabungan
    profile = ProfileReport(
        df,
        title='EDA Report - Forum 07',
        explorative=True
    )
    profile.to_file('EDA_Report_Text.html')
    print('EDA_Report_Text.html berhasil dibuat dan otomatis update sesuai df terakhir.')
except Exception as e:
    print('Gagal membuat EDA_Report_Text.html:', e)

Export report to file: 100%|██████████| 1/1 [00:00<00:00, 98.17it/s]

EDA_Report_Text.html berhasil dibuat dan otomatis update sesuai df terakhir.
